# Training SAM3 for Biomedical Image Segmentation

This notebook fine-tunes **SAM3** with **LoRA** on a biomedical instance-segmentation dataset
(e.g. cells / nuclei in microscopy images) and renders predictions as a **copper-on-black
instance overlay** — the segmented structures colored by their own intensity through the
`copper` colormap, everything else masked to black.

It reuses the exact machinery from the training scripts in this repo:

* `COCOSegmentDataset` and `SAM3TrainerNative` from [`train_sam3_lora_native.py`](train_sam3_lora_native.py)
  (model + LoRA + loss stack are built for us, identical to CLI training).
* `load_image_as_rgb` from [`image_utils.py`](image_utils.py) — so **TIFF** inputs
  (including 16-bit medical TIFFs) work out of the box.

### Expected dataset layout (COCO format)
```
<DATA_DIR>/
  train/
    _annotations.coco.json      # COCO instance annotations (bbox + segmentation)
    img_0001.tif                # images: jpg / png / tif / tiff ...
    ...
  valid/                        # optional
    _annotations.coco.json
    ...
```
This matches Roboflow-style COCO exports. The `file_name` field in the JSON decides the
extension, so TIFF datasets need no special handling.

## 1. Setup

Run this from the repository root so the local modules import correctly. A CUDA GPU is
strongly recommended — SAM3 is large. The first model build downloads `facebook/sam3`
weights from Hugging Face (needs internet + `huggingface-cli login` if the repo is gated).

In [ ]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Local modules (repo root must be the working directory)
from image_utils import load_image_as_rgb
from lora_layers import save_lora_weights, load_lora_weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cpu':
    print('WARNING: no GPU detected. Training SAM3 on CPU is impractically slow — '
          'use this notebook mainly to read the flow, or run on a GPU machine.')

## 2. Configuration

We start from the repo's `configs/full_lora_config.yaml` (which defines the LoRA setup and
loss weights) and override just the dataset path and a few training knobs here so we don't
have to edit the YAML. Point `DATA_DIR` at your dataset root.

In [ ]:
import yaml

BASE_CONFIG = 'configs/full_lora_config.yaml'

# ---- EDIT THESE ----
DATA_DIR   = 'data'          # root containing train/ (and optionally valid/)
NUM_EPOCHS = 30              # small biomedical sets converge fast; watch val loss
BATCH_SIZE = 2               # lower if you hit CUDA OOM
LR         = 5e-5
OUTPUT_DIR = 'outputs/sam3_biomed_lora'
# --------------------

with open(BASE_CONFIG) as f:
    config = yaml.safe_load(f)

config['training']['data_dir']      = DATA_DIR
config['training']['num_epochs']    = NUM_EPOCHS
config['training']['batch_size']    = BATCH_SIZE
config['training']['learning_rate'] = LR
config['training']['num_workers']   = 0      # 0 is safest inside notebooks on Windows
config['output']['output_dir']      = OUTPUT_DIR

# Persist the effective config so inference / CLI tools can reuse it later.
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
EFFECTIVE_CONFIG = str(Path(OUTPUT_DIR) / 'effective_config.yaml')
with open(EFFECTIVE_CONFIG, 'w') as f:
    yaml.safe_dump(config, f)

print('Effective config written to', EFFECTIVE_CONFIG)
config['training']

## 3. Load the dataset

`COCOSegmentDataset` handles COCO polygon/RLE masks and resizes everything to SAM3's
1008&times;1008 working resolution. Images load through `load_image_as_rgb`, so JPG/PNG/TIFF
all work.

In [ ]:
from train_sam3_lora_native import COCOSegmentDataset

train_ds = COCOSegmentDataset(data_dir=DATA_DIR, split='train')

val_ds = None
try:
    val_ds = COCOSegmentDataset(data_dir=DATA_DIR, split='valid')
    if len(val_ds) == 0:
        val_ds = None
except FileNotFoundError:
    print('No valid/ split found — training without validation.')

print(f'\nTrain images: {len(train_ds)}' + (f' | Val images: {len(val_ds)}' if val_ds else ''))

## 4. The copper-on-black overlay

This is the target look: each segmented structure is filled with its **own image intensity
passed through the `copper` colormap**, and everything outside the masks is black. We'll use
it both to preview ground truth now and to render predictions after training.

In [ ]:
from PIL import Image as PILImage

def _resize_mask(mask, size_hw):
    """Nearest-resize a bool/float HxW mask to (H, W)."""
    h, w = size_hw
    m = (np.asarray(mask) > 0.5).astype(np.uint8) * 255
    if m.shape != (h, w):
        m = np.array(PILImage.fromarray(m).resize((w, h), PILImage.NEAREST))
    return m > 127

def copper_overlay(pil_image, masks, edge_boost=0.0):
    """Render an instance overlay in the copper-on-black style.

    Args:
        pil_image : the RGB image the masks belong to.
        masks     : iterable of HxW bool/float masks (any resolution; resized to fit).
        edge_boost: 0..1, brighten mask borders to emphasise cell outlines.
    Returns HxWx3 float array in [0, 1].
    """
    gray = np.asarray(pil_image.convert('L'), dtype=np.float32) / 255.0
    H, W = gray.shape
    union = np.zeros((H, W), dtype=bool)
    for m in masks:
        union |= _resize_mask(m, (H, W))

    # Stretch intensity inside the masked region so faint structures stay visible.
    vals = gray[union]
    if vals.size:
        lo, hi = np.percentile(vals, 2), np.percentile(vals, 98)
        if hi > lo:
            gray = np.clip((gray - lo) / (hi - lo), 0, 1)

    rgb = plt.cm.copper(gray)[..., :3]
    out = rgb * union[..., None]

    if edge_boost > 0:
        from scipy.ndimage import binary_erosion
        edges = union & ~binary_erosion(union, iterations=1)
        out[edges] = np.clip(out[edges] + edge_boost, 0, 1)
    return out

def show_side_by_side(pil_image, masks, title='', **kw):
    ov = copper_overlay(pil_image, masks, **kw)
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(pil_image); ax[0].set_title('Input'); ax[0].axis('off')
    ax[1].imshow(ov);        ax[1].set_title(title or 'Segmentation overlay'); ax[1].axis('off')
    ax[1].set_facecolor('black')
    plt.tight_layout(); plt.show()
    return ov

### Preview a ground-truth sample
Sanity-check the dataset and see the target style before training.

In [ ]:
sample = train_ds[0]
raw = sample.raw_images[0]                      # PIL image at 1008x1008
gt_masks = [o.segment for o in sample.images[0].objects if o.segment is not None]
print(f'Sample has {len(gt_masks)} annotated instances.')
if gt_masks:
    _ = show_side_by_side(raw, gt_masks, title='Ground-truth masks (copper overlay)')

## 5. Build SAM3 + LoRA

`SAM3TrainerNative.__init__` builds the model, applies LoRA, and sets up the optimizer,
Hungarian matcher, and the full SAM3 loss stack — exactly as the CLI trainer does. We then
drive a compact training loop by hand so we get live loss curves in the notebook.

> First run downloads the SAM3 checkpoint; this can take a few minutes.

In [ ]:
from train_sam3_lora_native import SAM3TrainerNative

trainer = SAM3TrainerNative(EFFECTIVE_CONFIG, multi_gpu=False)

model       = trainer.model
optimizer   = trainer.optimizer
loss_wrapper = trainer.loss_wrapper
matcher     = trainer.matcher
unwrapped   = trainer._unwrapped_model
device      = trainer.device

## 6. Training loop

Mirrors the loop in `train_sam3_lora_native.py`: forward → build matched targets →
SAM3 loss → backward. Best/last LoRA weights are saved to `OUTPUT_DIR`.

In [ ]:
from torch.utils.data import DataLoader
from sam3.model.model_misc import SAM3Output
from sam3.train.loss.loss_fns import CORE_LOSS_KEY
from sam3.train.data.collator import collate_fn_api

def collate_fn(batch):
    return collate_fn_api(batch, dict_key='input', with_seg_masks=True)

def move_to_device(obj, dev):
    if isinstance(obj, torch.Tensor):
        return obj.to(dev)
    if isinstance(obj, list):
        return [move_to_device(x, dev) for x in obj]
    if isinstance(obj, tuple):
        return tuple(move_to_device(x, dev) for x in obj)
    if isinstance(obj, dict):
        return {k: move_to_device(v, dev) for k, v in obj.items()}
    if hasattr(obj, '__dataclass_fields__'):
        for fld in obj.__dataclass_fields__:
            setattr(obj, fld, move_to_device(getattr(obj, fld), dev))
        return obj
    return obj

def run_stage(input_batch):
    """One forward + loss for a collated batch. Returns the scalar loss tensor."""
    outputs_list = model(input_batch)
    find_targets = [unwrapped.back_convert(t) for t in input_batch.find_targets]
    for targets in find_targets:
        for k, v in targets.items():
            if isinstance(v, torch.Tensor):
                targets[k] = v.to(device)
    with SAM3Output.iteration_mode(outputs_list,
                                   iter_mode=SAM3Output.IterMode.ALL_STEPS_PER_STAGE) as it:
        for stage_outputs, stage_targets in zip(it, find_targets):
            for outputs in stage_outputs:
                outputs['indices'] = matcher(outputs, stage_targets)
                for aux in outputs.get('aux_outputs', []):
                    aux['indices'] = matcher(aux, stage_targets)
    return loss_wrapper(outputs_list, find_targets)[CORE_LOSS_KEY]

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=0, pin_memory=(device.type=='cuda'))
val_loader = (DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                         collate_fn=collate_fn, num_workers=0) if val_ds else None)

In [ ]:
out_dir = Path(OUTPUT_DIR); out_dir.mkdir(parents=True, exist_ok=True)
history = {'train': [], 'val': []}
best_val = float('inf')

for epoch in range(NUM_EPOCHS):
    model.train()
    ep_losses = []
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{NUM_EPOCHS}')
    for batch_dict in pbar:
        input_batch = move_to_device(batch_dict['input'], device)
        loss = run_stage(input_batch)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        ep_losses.append(loss.item()); pbar.set_postfix(loss=f'{loss.item():.3f}')
    train_loss = float(np.mean(ep_losses)) if ep_losses else 0.0
    history['train'].append(train_loss)

    val_loss = None
    if val_loader is not None:
        model.eval(); vl = []
        with torch.no_grad():
            for batch_dict in val_loader:
                input_batch = move_to_device(batch_dict['input'], device)
                vl.append(run_stage(input_batch).item())
        val_loss = float(np.mean(vl)); history['val'].append(val_loss)

    save_lora_weights(model, str(out_dir / 'last_lora_weights.pt'))
    tag = f'train {train_loss:.4f}' + (f' | val {val_loss:.4f}' if val_loss is not None else '')
    if val_loss is not None and val_loss < best_val:
        best_val = val_loss
        save_lora_weights(model, str(out_dir / 'best_lora_weights.pt'))
        tag += '  <- new best'
    elif val_loss is None:
        save_lora_weights(model, str(out_dir / 'best_lora_weights.pt'))
    print(f'Epoch {epoch+1}: {tag}')
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print('\nDone. Weights saved in', OUTPUT_DIR)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(history['train'], label='train')
if history['val']:
    plt.plot(history['val'], label='val')
plt.xlabel('epoch'); plt.ylabel('SAM3 loss'); plt.legend(); plt.title('Training curve')
plt.grid(alpha=0.3); plt.show()

## 7. Inference &rarr; copper overlay

Run the trained model on an image and render the predicted instance masks in the same style.
We filter by confidence and apply NMS to keep the overlay clean (one mask per structure).

In [ ]:
from torchvision.ops import nms
from torchvision.transforms import v2
from sam3.train.data.sam3_image_dataset import Datapoint, Image, FindQueryLoaded, InferenceMetadata

RES = 1008
_infer_tf = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
])

@torch.no_grad()
def segment(image_path, prompt='cell', conf=0.5, nms_iou=0.5, max_instances=400):
    """Return (pil_image, list_of_bool_masks) for the given image."""
    model.eval()
    pil = load_image_as_rgb(image_path)
    orig_w, orig_h = pil.size
    resized = pil.resize((RES, RES), PILImage.BILINEAR)
    img_obj = Image(data=_infer_tf(resized), objects=[], size=(RES, RES))
    query = FindQueryLoaded(query_text=prompt, image_id=0, object_ids_output=[],
                            is_exhaustive=True, query_processing_order=0,
                            inference_metadata=InferenceMetadata(
                                coco_image_id=0, original_image_id=0, original_category_id=0,
                                original_size=(orig_h, orig_w), object_id=-1, frame_index=-1))
    dp = Datapoint(find_queries=[query], images=[img_obj], raw_images=[resized])
    input_batch = move_to_device(collate_fn([dp])['input'], device)

    outputs = model(input_batch)[-1]
    scores = outputs['pred_logits'].sigmoid()[0].max(dim=1).values   # [Q]
    boxes  = outputs['pred_boxes'][0]                                # [Q,4] cxcywh norm
    masks  = outputs.get('pred_masks')
    masks  = masks[0] if masks is not None else None                 # [Q,h,w]

    keep = (scores > conf).nonzero(as_tuple=True)[0]
    if keep.numel() == 0:
        print('No predictions above confidence', conf); return pil, []
    cx, cy, w, h = [boxes[keep, i] for i in range(4)]
    xyxy = torch.stack([cx - w/2, cy - h/2, cx + w/2, cy + h/2], dim=1)
    keep = keep[nms(xyxy, scores[keep], nms_iou)][:max_instances]
    print(f'{len(keep)} instances after confidence + NMS')

    out_masks = []
    if masks is not None:
        for i in keep.tolist():
            out_masks.append((masks[i].sigmoid() > 0.5).cpu().numpy())
    return pil, out_masks

In [ ]:
# Point this at any image (jpg / png / tif). Defaults to the first validation/training image.
src = val_ds if val_ds is not None else train_ds
first_id = src.image_ids[0]
TEST_IMAGE = str(src.split_dir / src.images[first_id]['file_name'])
print('Segmenting', TEST_IMAGE)

pil, pred_masks = segment(TEST_IMAGE, prompt='cell', conf=0.5, nms_iou=0.5)
if pred_masks:
    _ = show_side_by_side(pil, pred_masks, title='Predicted masks (copper overlay)', edge_boost=0.15)

In [ ]:
# Save a standalone copper-on-black overlay image (black background, no axes).
if pred_masks:
    overlay = copper_overlay(pil, pred_masks, edge_boost=0.15)
    save_path = str(out_dir / 'prediction_overlay.png')
    plt.imsave(save_path, overlay)
    print('Saved', save_path)

## Notes & next steps

* **Prompt matters.** SAM3 is promptable — set `prompt=` in `segment(...)` to the structure
  you annotated (`'cell'`, `'nucleus'`, `'gland'`, ...). It should match the class name /
  query text used during training for best results.
* **Overlay controls.** `conf` and `nms_iou` trade recall vs. a cleaner overlay; `edge_boost`
  brightens cell borders. The colormap is `copper` — swap `plt.cm.copper` in `copper_overlay`
  for `magma`, `viridis`, etc. if you prefer a different look.
* **Full metrics.** For mAP / cgF1 with NMS, run `validate_sam3_lora.py` with `--weights`
  pointing at `outputs/sam3_biomed_lora/best_lora_weights.pt` and the same config.
* **OOM?** Lower `BATCH_SIZE`, or reduce LoRA scope in the YAML (`configs/light_lora_config.yaml`
  / `minimal_lora_config.yaml`).